<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week9/Day3/Daily_Challenge/Agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [1]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.8 MB/s eta 0:00:00


## 1) Define KB

In [2]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [6]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {"query": {"type": "string", "description": "The search query to look up in the knowledge base."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {
            "type": "string",
            "description": "The operation to perform: 'add' or 'multiply'.",
            "nullable": True
        }
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [7]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
)

print("Model ready:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/smolagents/models.py:933: FutureWarning: The 'model_id' parameter will be required in version 2.0.0. Please update your code to pass this parameter to avoid future errors. For now, it defaults to 'HuggingFaceTB/SmolLM2-1.7B-Instruct'.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [9]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI that uses tools to answer questions."
    ),
)

print(agent)

## 5) Test queries

In [10]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-1.7B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12, 'b': 30, 'op': 'add'}                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Unknown tool agentic_ai_loop, should be one of: kb_lookup, math_tool, final_answer.

[Step 2: Duration 6.86 seconds| Input tokens: 2,519 | Output tokens: 385]

Reached max steps.

[Step 3: Duration 3.15 seconds| Input tokens: 3,114 | Output tokens: 461]

Answer: An agentic AI loop is a type of loop in artificial intelligence where an AI system is able to perform a task or set of tasks, and then use the results of those tasks to inform and improve its performance on the same task or other tasks. This creates a continuous cycle of learning and improvement, where the AI system is able to adapt and refine its performance over time.
